# Enterprise Migration Copilot 
# Fine-tuning: SQL/HiveQL/PL-SQL → PySpark Migration Models

QLoRA fine-tuning on the enterprise migration dataset (1,312 validated pairs).
Run once per model, changing `MODEL_NAME` in Cell 2 each time.

**Setup:** Runtime → T4 GPU · Add `HF_TOKEN` to Colab Secrets (🔑 sidebar)

- Session 1: `deepseek-ai/deepseek-coder-1.3b-instruct`
- Session 2: `Qwen/Qwen2.5-Coder-1.5B-Instruct`
- Session 3: `microsoft/Phi-3.5-mini-instruct`


In [ ]:
# Cell 1 — Install dependencies (no bitsandbytes)
!pip install -q transformers==4.46.0 datasets peft==0.13.0 trl==0.11.0 accelerate==0.34.0 huggingface_hub
!pip install -q numpy==1.26.4

import os
os.kill(os.getpid(), 9)

In [ ]:
# Cell 2 — Configuration
# ====================================================
# CHANGE MODEL_NAME FOR EACH SESSION:
# Session 1: "deepseek-ai/deepseek-coder-1.3b-instruct"
# Session 2: "Qwen/Qwen2.5-Coder-1.5B-Instruct"
# Session 3: "microsoft/Phi-3.5-mini-instruct"
# ====================================================
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

HF_USERNAME = "praveends"
DATASET_REPO = f"{HF_USERNAME}/enterprise-migration-dataset"

# Auto-generate output repo name from model name
model_short = MODEL_NAME.split('/')[-1].lower().replace('.', '-')
OUTPUT_REPO = f"{HF_USERNAME}/migration-copilot-{model_short}"

print(f"Model:   {MODEL_NAME}")
print(f"Dataset: {DATASET_REPO}")
print(f"Output:  {OUTPUT_REPO}")

In [ ]:
# Cell 3 — HuggingFace login
# Token stored in Colab Secrets — never appears in notebook
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to HuggingFace")

In [ ]:
# Cell 4 — Load and format dataset
from datasets import load_dataset

dataset = load_dataset(DATASET_REPO, data_files="train.jsonl", split="train")
print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

def format_prompt(example):
    """
    Format each training pair as an instruction-following prompt.
    This exact format is used at inference time by the migrator agent.
    """
    source_lang = example.get('source_language', 'sql').upper()
    difficulty = example.get('difficulty', 'medium')
    source_code = example.get('source_code', '')
    pyspark_code = example.get('pyspark_code', '')

    prompt = f"""### Instruction:
Convert the following {source_lang} code to PySpark.
Difficulty: {difficulty}

### Input:
{source_code}

### Response:
{pyspark_code}
### End"""

    return {"text": prompt}

dataset = dataset.map(format_prompt, load_from_cache_file=False)  # force re-map
# 95/5 train/eval split
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print("\nSample prompt (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
# Cell 5 — Load model in float16 (no quantization needed for 1.3B on T4)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

print(f"\nModel loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 6 — Apply LoRA adapters (no quantization prep needed)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 7 — Fine-tune with SFTTrainer
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

try:
    training_args = SFTConfig(
        output_dir=f"/tmp/{model_short}-finetuned",
        num_train_epochs=3,
        per_device_train_batch_size=1,       # reduced from 4
        gradient_accumulation_steps=8,       # increased to compensate
        warmup_steps=50,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        optim="adamw_torch",
        load_best_model_at_end=True,
        report_to="none",
        max_seq_length=512,                  # reduced from 1024
        dataset_text_field="text",
    )
    print("SFTConfig OK")

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
    )
    print("SFTTrainer OK")

    print(f"Training on {len(train_dataset)} examples")
    trainer.train()
    print("Training complete!")

except Exception as e:
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 8 — Push fine-tuned model to HuggingFace
print(f"Pushing model to: {OUTPUT_REPO}")
trainer.model.push_to_hub(OUTPUT_REPO)
tokenizer.push_to_hub(OUTPUT_REPO)
print(f"\nDone! Model live at:")
print(f"https://huggingface.co/{OUTPUT_REPO}")

In [ ]:
# Cell 9 — Quick inference test
from transformers import StoppingCriteria, StoppingCriteriaList
import torch

class StopOnSequence(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids
        self.len = len(stop_token_ids)

    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] < self.len:
            return False
        return input_ids[0][-self.len:].tolist() == self.stop_token_ids

stop_ids = tokenizer("### End", add_special_tokens=False).input_ids
print("Stop token IDs:", stop_ids)
stopping_criteria = StoppingCriteriaList([StopOnSequence(stop_ids)])

test_prompt = """### Instruction:
Convert the following SQL code to PySpark.
Difficulty: medium

### Input:
SELECT customer_id, SUM(amount) AS total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY total DESC;

### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    stopping_criteria=stopping_criteria,
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
generated = response[len(test_prompt):].strip()

# Clean up ### End if it appears in output
generated = generated.replace("### End", "").replace("###", "").strip()

print("=" * 60)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 60)
print(generated)
print("=" * 60)

## Results

| Model | Train Loss | Eval Loss | HF Repo |
|---|---|---|---|
| deepseek-coder-1.3b-instruct | 0.258 | 0.329 | migration-copilot-deepseek-coder-1-3b-instruct |
| Qwen2.5-Coder-1.5B-Instruct | 0.307 | 0.344 | migration-copilot-qwen2-5-coder-1-5b-instruct |
| Phi-3.5-mini-instruct | — | — | migration-copilot-phi-3-5-mini-instruct |